# CondVQA - Conditional Visual Question Answering with Integrated Gradients

**Requirements:** Google Colab with GPU runtime (Runtime > Change runtime type > GPU)

## Section 1: Setup

In [ ]:
# Install numpy (required specific version)
!pip uninstall -y numpy
!pip install numpy==1.26.4

print("\n!! Please restart runtime: Runtime > Restart runtime !!")
print("After restart, skip this cell and run the next one.")

In [ ]:
# Clone repo and install dependencies
!git clone -b clean https://github.com/rdgbrandon/CondVQA.git
%cd CondVQA
!pip install -q -r requirements.txt

In [ ]:
# Load models
import sys
sys.path.insert(0, './src')

from src import load_model, vqa_interpret, conditional_query_vqa_interpret, text_vqa_interpret
import torch
import gc

model, processor = load_model()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: GPU not available. Enable GPU in Runtime > Change runtime type")

## Section 2: Image Analysis

Upload an image, run VQA with Integrated Gradients attribution heatmaps and text attribution.

In [ ]:
# Upload image
gc.collect()
torch.cuda.empty_cache()

from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"Uploaded: {image_path}")

In [ ]:
# Image VQA with Integrated Gradients attribution
questions = [
    "What is in this image?",
    "What color is the main object?"
]

vqa_interpret(
    image_path=image_path,
    questions=questions,
    model=model,
    processor=processor,
    show_top_k=10
)

In [ ]:
# Text attribution - which words in the question matter most?
text_results = text_vqa_interpret(
    image_path=image_path,
    questions=questions,
    model=model,
    processor=processor,
    mode='both',
    n_steps=10,
    show_visualizations=True
)

## Section 3: Video Analysis (Conditional VQA)

Upload a video, ask conditional questions like *"When there is a dog, what color is it?"*

The system uses a text LLM to parse the question into condition + question, scans frames for the condition, then runs IG attribution on matching frames.

In [ ]:
# Upload video
gc.collect()
torch.cuda.empty_cache()

from google.colab import files
uploaded_video = files.upload()
video_path = list(uploaded_video.keys())[0]
print(f"Uploaded: {video_path}")

In [ ]:
# Conditional VQA with Integrated Gradients
conditional_questions = [
    "When you see birds, how many do you see?"
]

conditional_results = conditional_query_vqa_interpret(
    video_path=video_path,
    questions=conditional_questions,
    model=model,
    processor=processor,
    fps_sample=1,
    confidence_threshold=0.5,
    aggregation_method='most_confident',
    show_visualizations=True,
    save_results=True,
    include_text_attribution=True
)

# Summary
for question, result in conditional_results.items():
    print(f"\nQ: {question}")
    if 'error' in result:
        print(f"  Error: {result['error']}")
        continue
    parsed = result['query_info']['parsed']
    print(f"  Type: {parsed['type']}")
    if parsed['type'] == 'conditional':
        print(f"  Condition: {parsed['frame_condition']}")
        print(f"  Frames matched: {result['query_info']['num_matching_frames']}/{result['query_info']['total_frames']}")
    print(f"  Answer: {result['prediction']}")
    print(f"  Confidence: {result['confidence']:.2%}")